In [1]:
%reload_ext autoreload
%autoreload 2

from MBN_Res_Constrn import MBN_RC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
#put network you are using eg, full, node removed
name='RE'

#put itr as number of iterations you want
itr=800
#to get and store iterations data
c=600

#to store iteration data
df_itr=pd.DataFrame()   

#to store local order data
df_loc_data = pd.DataFrame()

#Lists to store transition time and time spent in sync state
trans_time=[]
state_time=[]


while c<itr:
    print("Current iteration is",c)
    mbn = MBN_RC(nepochs=40000, 
                 dt=0.05, 
                 lambda_o=2.86, 
                 alpha=0.01,
                 beta=0.002,
                 plot_bifurcation=False)
    
    mbn.run_model()

    #df to store 1 iteration data
    df=pd.DataFrame(mbn.GLOBAL_ORDER_VERBOSE)
    #concatenating each iteration as a column
    df_itr=pd.concat([df_itr,df], axis=1)

    #changing headers to number of iterations
    df_itr.columns=range(0,df_itr.shape[1])
    #storing dataframe in csv file
    df_itr.to_csv("data6.csv")

    #smoothing the dataframe
    df_smooth=df.rolling(window=800, center=True).mean()

    #defining time and timesteps for data
    time_steps = list(range(0,df_itr.shape[0]))   
    time=np.multiply(time_steps,mbn.dt)

    #dropping the NaN values
    df_na = df_smooth.dropna()
    

    #defining time and time steps for df_na data
    #can drop these 2 lines
    time_steps_na = list(range(0,df_na.shape[0])) 
    time_na=np.multiply(time_steps_na,mbn.dt)

    
    data = np.array(df_na)
    #Looping one cycle of finding transition and time spent in sync state
    while True:
        
        index_arr=np.where(data >= 0.4)[0]

        #Finding if iteration has a transition
        if index_arr.size > 0:
            upper_crossing = index_arr[0]
           #finding transition time thresholds     
            l1 = np.where(data >= 0.1)[0]
            l2=np.where(data <= 0.101)[0]
            low_intersection = np.intersect1d(l1, l2)
            #making sure lower thresholds are for 1st transition
            low_upd =low_intersection[low_intersection<upper_crossing]

            lower_crossing=low_upd[-1]
        
            t=(upper_crossing-lower_crossing)*mbn.dt
            trans_time.append(t)
            #trans_df=pd.concat([trans_df, df_na[i]], axis=1)
            #trans_df stores the iterations which have transitions

            #Extracting local order data for above iteration
            df_dum = pd.DataFrame()
            #defining local order data timestep thresholds
        
            #checking where it crossed 0.3
            m_loc=np.where(data >= 0.3)[0][0]
        

        
            lt_loc = m_loc - 3000
            ut_loc = m_loc + 2000

            #extracting data from df_loc
            header_list=list(range(lt_loc,ut_loc))
            #filtering those columns which lie in between 0 to mbn.nepochs
            header_list_f = [x for x in header_list if 0 <= x < mbn.nepochs]
            df_dum = mbn.df_loc[header_list_f]
            #changing column numbers so that they be concated 1 below other
            df_dum.columns = range(0,df_dum.shape[1])
        

            #creating multi-index dataframe
            index=[[c]*426,list(range(0,426))]
            df_dum=df_dum.set_index(index)
            df_loc_data = pd.concat([df_loc_data,df_dum])
        

        
            #checking for another transition
            fwd_data=np.array(data[upper_crossing:])
        

            #upper_crs=np.where(fwd_data >= 0.45)[0][-1]
            check=np.where(fwd_data <= 0.2)[0]

            if check.size > 0:
                ind=np.where(fwd_data <= 0.1)[0]
                if ind.size > 0:

                    lower_crs = ind[0]
                    lower_crs_up = lower_crs+upper_crossing
                    st_time=(lower_crs_up-lower_crossing)*mbn.dt
                    state_time.append(st_time)
                    #creating data for other cycle of transition and state time
                    data= np.array(data[lower_crs_up:])
                #trans_2df=pd.concat([trans_2df, df_na[i]], axis=1)
                else:
                    
                    break
                    

            else:
                #continue
                
                break
                

        else:
                #data_no_trans=pd.concat([data_no_trans, df_na[i]], axis=1)
            
            break
    
    #increasing count by 1
    c+=1
 

    
#define network for which you are calculatingab

df_tt = pd.DataFrame(trans_time,columns=[name])
df_st=  pd.DataFrame(state_time,columns=[name])


print(np.array(trans_time))
print(np.array(state_time))
print(f'The number of iterations with transition is {len(trans_time)}')    
print(f'The number of iterations with more than 1 transition is {len(state_time)}') 


df_tt.to_csv('tt6_800.csv')
df_st.to_csv('st6_800.csv')
df_loc_data=df_loc_data.astype(np.float32)
df_loc_data.to_pickle('l800.bz2',compression='bz2')

Current iteration is 600
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.57016388579491
Current iteration is 601
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.19746044161196255
Current iteration is 602
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


...

/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.4258119416776702
Current iteration is 647
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.5934500063435532
Current iteration is 648
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K
